# SlaTriage — eval A–E–G sau khi đã train

1. GPU T4, Internet on.
2. Add data: `labels_v1.jsonl` + folder `adapters` (hoặc unzip `slatriage_adapters.zip`).
3. Run All. Tải `reports/` khi xong.

In [ ]:
import os, shutil
from pathlib import Path

assert __import__('torch').cuda.is_available(), 'Enable GPU T4 + Restart'
REPO = 'https://github.com/kham3465/Qu-n-Tr-An-Ninh-M-ng.git'
ROOT = Path('/kaggle/working/slatriage')
if not (ROOT / '.git').exists():
    !git clone --depth 1 {REPO} {ROOT}
else:
    !git -C {ROOT} fetch origin master
    !git -C {ROOT} reset --hard origin/master

work = Path('/kaggle/working')
gold_hits = list(Path('/kaggle/input').glob('**/labels_v1.jsonl')) + list(work.glob('**/labels_v1.jsonl'))
assert gold_hits, 'Upload labels_v1.jsonl'
shutil.copy2(gold_hits[0], work / 'labels_v1.jsonl')

ad = work / 'adapters'
if not (ad / 'R' / 'adapter_config.json').exists():
    hits = list(Path('/kaggle/input').glob('**/R/adapter_config.json'))
    if hits:
        src = hits[0].parent.parent
        if src.resolve() != ad.resolve():
            if ad.exists():
                shutil.rmtree(ad)
            shutil.copytree(src, ad)
    else:
        zips = list(Path('/kaggle/input').glob('**/slatriage_adapters.zip')) + list(work.glob('*.zip'))
        assert zips, 'Upload adapters or slatriage_adapters.zip'
        import zipfile
        ad.mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(zips[0]).extractall(ad)
print('gold', work / 'labels_v1.jsonl')
print('adapters', ad)
os.chdir(ROOT)

In [ ]:
!pip install -q -r requirements-train.txt
!python scripts/kaggle_eval.py